# 6. NumPy

Uma frase para começar: **o pandas não é rápido. O NumPy é.**

Toda coluna de um DataFrame é, por dentro, um array do NumPy. É de lá que vem a
velocidade, é de lá que vem a regra de um tipo por coluna, e é de lá que vem metade do
vocabulário que você já usa sem saber: máscara booleana, `axis`, `NaN`.

Esta aula é sobre essa camada.

In [ ]:
import time

import numpy as np
import pandas as pd

print("numpy:", np.__version__, "| pandas:", pd.__version__)

numpy: 2.5.2 | pandas: 3.0.5


In [ ]:
m = np.ones((3,4))
display(m.T + np.array([1,2,3]))
display(m + np.array([1,2,3,4]))

array([[2., 3., 4.],
       [2., 3., 4.],
       [2., 3., 4.],
       [2., 3., 4.]])

array([[2., 3., 4., 5.],
       [2., 3., 4., 5.],
       [2., 3., 4., 5.]])

---
## 1. A diferença

Uma lista Python guarda **ponteiros** para objetos espalhados na memória. Cada `int` é um
objeto completo, com tipo e contador de referências.

Um `ndarray` guarda os **números**, lado a lado, todos do mesmo tipo.

In [ ]:
import sys as _sys

lista = [1, 2, 3, 4, 5]
arr = np.array([1, 2, 3, 4, 5])

print("lista:", _sys.getsizeof(lista), "bytes de estrutura +", _sys.getsizeof(1), "por número")
print("array:", arr.nbytes, "bytes de dado |", arr.dtype, "| contíguo:", arr.flags["C_CONTIGUOUS"])

lista: 104 bytes de estrutura + 28 por número
array: 40 bytes de dado | int64 | contíguo: True


Daí vem tudo. Um tipo só significa que a soma não precisa perguntar *"que tipo é
este?"* a cada elemento. Memória contígua significa que o processador lê em bloco, e o
código que percorre o array é compilado, não interpretado.

---
## 2. O preço, medido

In [ ]:
n = 1_000_000
rng = np.random.default_rng(0)
arr = rng.random(n)
lista = arr.tolist()
serie = pd.Series(arr)


def cronometrar(rotulo, funcao):
    inicio = time.perf_counter()
    resultado = funcao()
    print(f"  {rotulo:22} {(time.perf_counter() - inicio) * 1000:8.1f} ms   soma={resultado:.4f}")


cronometrar("for em Python", lambda: sum(v for v in lista))
cronometrar("pandas .sum()", lambda: serie.sum())
cronometrar("numpy .sum()", lambda: arr.sum())

  for em Python              34.0 ms   soma=500159.2565
  pandas .sum()               3.4 ms   soma=500159.2565
  numpy .sum()                0.6 ms   soma=500159.2565


O pandas e o NumPy rodam praticamente no mesmo tempo, porque **são a mesma conta**.
O `.sum()` da Series delega para o array que está lá dentro.

Guarde a regra prática: **se você escreveu um `for` sobre as linhas de um DataFrame,
quase sempre existe uma forma vetorizada, e ela é uma ordem de grandeza mais rápida.**

---
## 3. `ndarray`: forma e tipo

Duas propriedades explicam quase tudo: `shape` e `dtype`.

In [ ]:
a = np.arange(12)
print(a, "\n shape:", a.shape, "| dtype:", a.dtype, "| ndim:", a.ndim, "| size:", a.size)

m = a.reshape(3, 4)
print("\n", m, "\n shape:", m.shape, "| ndim:", m.ndim)
print("\n reshape(-1, 2):", a.reshape(-1, 2).shape, "  -1 = calcule você")
print(" transposta:", m.T.shape, "| achatada:", m.ravel().shape)

[ 0  1  2  3  4  5  6  7  8  9 10 11] 
 shape: (12,) | dtype: int64 | ndim: 1 | size: 12

 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]] 
 shape: (3, 4) | ndim: 2

 reshape(-1, 2): (6, 2)   -1 = calcule você
 transposta: (4, 3) | achatada: (12,)


In [ ]:
# um dtype para o array inteiro: misturar promove tudo
print(np.array([1, 2, 3]).dtype)
print(np.array([1, 2, 3.5]).dtype, "<- um float promoveu os inteiros")
print(np.array([1, 2, "a"]).dtype, "<- e um texto promove tudo a string")
print(np.array([1.9, 2.9]).astype(int), "<- astype trunca, não arredonda")

int64
float64 <- um float promoveu os inteiros
<U21 <- e um texto promove tudo a string
[1 2] <- astype trunca, não arredonda


**✏️ Exercícios rápidos**


1. Crie `np.arange(24)` e transforme numa matriz 4×6. Depois em 6×4, usando `-1`.
2. Qual o `dtype` de `np.array([1, 2, 3.0])`? E de `np.array([True, 1])`? Por quê?
3. Converta `np.array([9.9, 2.1])` para inteiro. O resultado arredonda ou trunca?

In [ ]:
# 1)
arr = np.arange(24)
display(arr.reshape(4,6))
display(arr.reshape(-1,4))
# 2)

display(np.array([1, 2, 3.0]).dtype)
display(np.array([True, 1]).dtype)

# 3)

display(np.ceil(np.array([9.9, 2.1])).astype(int))

array([[ 0,  1,  2,  3,  4,  5],
       [ 6,  7,  8,  9, 10, 11],
       [12, 13, 14, 15, 16, 17],
       [18, 19, 20, 21, 22, 23]])

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11],
       [12, 13, 14, 15],
       [16, 17, 18, 19],
       [20, 21, 22, 23]])

dtype('float64')

dtype('int64')

array([10,  3])

---
## 4. Vetorização

Escrever a operação sobre o array inteiro, em vez de elemento a elemento.

In [ ]:
a = np.array([1, 2, 3, 4])
print(a + 10, a * 2, a ** 2, a / 2)

print(np.sqrt([1, 4, 9]), np.round([1.27, 3.51], 1), np.abs([-1, 2]))
print(np.clip([1, 1, 5, 10, 7], 2, 8), "<- prende no intervalo")

[11 12 13 14] [2 4 6 8] [ 1  4  9 16] [0.5 1.  1.5 2. ]
[1. 2. 3.] [1.3 3.5] [1 2]
[2 2 5 8 7] <- prende no intervalo


### Broadcasting

Quando as formas são diferentes, o NumPy **estica** a menor em vez de dar erro. É por isso
que `a + 10` funciona sem você repetir o 10 quatro vezes.

In [ ]:
m = np.arange(6).reshape(2, 3)
print(m, "\n")
print(m + np.array([10, 20, 30]), "  <- (2,3) + (3,): a linha se repete\n")
print(m + np.array([[100], [200]]), "  <- (2,3) + (2,1): a coluna se repete\n")

try:
    m + np.array([1, 2])
except ValueError as e:
    print("ValueError:", str(e)[:70])

[[0 1 2]
 [3 4 5]] 

[[10 21 32]
 [13 24 35]]   <- (2,3) + (3,): a linha se repete

[[100 101 102]
 [203 204 205]]   <- (2,3) + (2,1): a coluna se repete

ValueError: operands could not be broadcast together with shapes (2,3) (2,) 


**✏️ Exercícios rápidos**


1. Com `a = np.array([2, 4, 6])`, calcule `a * 10`, `a ** 2` e `a / 4`.
2. Some `np.arange(6).reshape(2, 3)` com `np.array([1, 2, 3])`. Qual linha mudou?
3. Tente somar a mesma matriz com `np.array([1, 2])`. Leia o erro e diga por quê.

In [ ]:
# 1)
a = np.array([2, 4, 6])
print(a * 10, a ** 2, a / 4)

# 2)
np.arange(6).reshape(2, 3) + np.array([1,2,3])

# 3)
np.arange(6).reshape(2, 3) + np.array([1,2])


[20 40 60] [ 4 16 36] [0.5 1.  1.5]


ValueError: operands could not be broadcast together with shapes (2,3) (2,) 

---
## 5. Indexação booleana

A máscara do pandas é isto aqui, um andar abaixo.

In [ ]:
a = np.array([10, 20, 30, 40, 50])

mascara = a > 25
print(mascara, "|", mascara.dtype)
print(a[mascara], "<- só os marcados como True")
print(a[(a > 15) & (a < 45)], "<- & e |, com parênteses, igual ao pandas")
print(a[[0, 3]], "<- fancy indexing: lista de posições")

a[a > 30] = 0                      # atribuir pela máscara altera no lugar
print(a)

[False False  True  True  True] | bool
[30 40 50] <- só os marcados como True
[20 30 40] <- & e |, com parênteses, igual ao pandas
[10 40] <- fancy indexing: lista de posições
[10 20 30  0  0]


In [ ]:
# np.where: o if/else vetorizado
valores = np.array([100, 800, 250, 1500])
print(np.where(valores > 500, "alto", "baixo"))
print(np.where(valores > 500, valores * 0.9, valores), "<- pode devolver valores, não só rótulos")

['baixo' 'alto' 'baixo' 'alto']
[ 100.  720.  250. 1350.] <- pode devolver valores, não só rótulos


**✏️ Exercícios rápidos**


1. De `a = np.array([5, 12, 7, 30, 1])`, pegue só os maiores que 6.
2. Quantos elementos de `a` estão entre 5 e 20? (some a máscara: `True` vale 1)
3. Troque por zero todos os valores acima de 10, sem `for`.

In [ ]:
a = np.array([5, 12, 7, 30, 1])

# 1)

display(
    a[a > 6]
)

# 2)
maior_5 = a > 5
menor_20 = a < 20
display(
     (maior_5 & menor_20).sum()
)

# 3)

display(
    np.where(
        a > 10,
        a, 0
    )
)

array([12,  7, 30])

np.int64(2)

array([ 0, 12,  0, 30,  0])

### Adendo: `np.where` numa coluna de DataFrame

Na aula 5 você atribuiu por máscara assim:

```python
d.loc[d["valor"] > 500, "faixa"] = "alto"
```

Aquilo escreve **só nas linhas que casam**: as outras ficam `NaN`. Para preencher as
duas pontas você teria que repetir a atribuição com a condição invertida.

O `np.where` faz os dois lados numa expressão só, e aceita uma coluna do pandas
diretamente, porque por dentro ela **é** um array.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "produto": ["notebook", "cadeira", "mouse", "mesa", "monitor"],
    "valor":   [3200.0, 890.0, 150.0, np.nan, 1450.0],
})

# o jeito da aula 5: escreve só onde a máscara é True
d = df.copy()
d.loc[d["valor"] > 500, "faixa"] = "alto"
print(d["faixa"].tolist(), " <- NaN onde não casou")

# np.where: os dois lados de uma vez
d["faixa"] = np.where(d["valor"] > 500, "alto", "baixo")
print(d["faixa"].tolist())

['alto', 'alto', nan, nan, 'alto']  <- NaN onde não casou
['alto', 'alto', 'baixo', 'baixo', 'alto']


> ⚠️ Repare no `NaN` da mesa: para o `np.where` ele caiu em `"baixo"`, porque
> `NaN > 500` é `False`. Nulo **não** é uma terceira resposta: ele some dentro do
> `else`. Se isso importa, trate o nulo antes, ou encadeie:

In [ ]:
d = df.copy()
d["faixa"] = np.where(d["valor"].isna(), "sem valor",
                      np.where(d["valor"] > 500, "alto", "baixo"))
d

,produto,valor,faixa
0,notebook,3200.0,alto
1,cadeira,890.0,alto
2,mouse,150.0,baixo
3,mesa,NaN,sem valor
4,monitor,1450.0,alto


**✏️ Exercícios rápidos**

1. Com `np.where`, marque `"caro"`/`"barato"` num corte de 1000 no `valor`.
2. Aplique 10% de desconto **só** nos acima de 1000, deixando os outros como estão
   (dica: o segundo e o terceiro argumento podem ser contas).
3. Faça o mesmo do item 2 com `.loc`. Qual das duas versões você acha mais legível?

In [ ]:
# 1)
d = df.copy()

d["faixa"] = np.where(d["valor"] > 1000, "caro", "barato")
display(d)

# 2)
d["descontado"] = np.where(d["valor"] > 1000, d["valor"] * 0.9, d["valor"])
display(d)


# 3)
d = df.copy()
d["descontado"] = d["valor"]
d.loc[d["valor"] > 1000, "descontado"] = d["valor"] * 0.9
display(d)

,produto,valor,faixa
0,notebook,3200.0,caro
1,cadeira,890.0,barato
2,mouse,150.0,barato
3,mesa,NaN,barato
4,monitor,1450.0,caro


,produto,valor,faixa,descontado
0,notebook,3200.0,caro,2880.0
1,cadeira,890.0,barato,890.0
2,mouse,150.0,barato,150.0
3,mesa,NaN,barato,NaN
4,monitor,1450.0,caro,1305.0


,produto,valor,descontado
0,notebook,3200.0,2880.0
1,cadeira,890.0,890.0
2,mouse,150.0,150.0
3,mesa,NaN,NaN
4,monitor,1450.0,1305.0


**✏️ Exercícios rápidos**

1. De `a = np.arange(12)`, os elementos maiores que 3 **e** menores que 9.
2. Os que são menores que 3 **ou** maiores que 9.
3. Os que **não** são pares, usando `~`.
4. Quantos atendem à condição do item 1? (some a máscara: `True` vale 1)
5. Tire os parênteses do item 1 e leia o erro.

In [ ]:
a = np.arange(12)

# 1)


# 2)


# 3)


# 4)


# 5)

---
## 6. `axis`: a fonte de confusão

`axis` diz qual eixo **desaparece**, não qual sobra.

In [ ]:
m = np.arange(12).reshape(3, 4)
print(m, "\n")
print("m.sum()        ->", m.sum(), "   tudo num escalar")
print("m.sum(axis=0)  ->", m.sum(axis=0), "   some as LINHAS: sobra uma por coluna")
print("m.sum(axis=1)  ->", m.sum(axis=1), "      some as COLUNAS: sobra uma por linha")
print("\nshape:", m.shape, "->", m.sum(axis=0).shape, "ou", m.sum(axis=1).shape)
print("\nmean, min, max, std, argmax:", m.mean(), m.min(), m.max(), round(m.std(), 2), m.argmax())

É a mesma convenção do pandas: `df.sum(axis=0)` soma cada coluna, `axis=1` soma cada
linha.

**✏️ Exercícios rápidos**


1. Em `m = np.arange(20).reshape(4, 5)`, some cada coluna. Quantos valores saem?
2. Agora some cada linha. E qual a média geral?
3. Sem olhar a resposta antes: qual o `shape` de `m.sum(axis=1)`? Confira depois.

In [ ]:
m = np.arange(20).reshape(4, 5)

# 1)


# 2)


# 3)

---
## 7. Nulos, e por que NaN é float

Aquela regra que você já viu no pandas: *coluna de inteiros com nulo vira `float64`*,
nasce aqui.

In [ ]:
print("NaN é do tipo:", type(np.nan).__name__)
print(np.array([1, 2, 3]).dtype, "->", np.array([1, np.nan, 3]).dtype,
      " NaN não cabe num bloco de inteiros")

a = np.array([1.0, np.nan, 3.0])
print("\na.sum()   ->", a.sum(), "  um único NaN contamina a conta inteira")
print("np.nansum ->", np.nansum(a), "  a família nan* ignora os nulos")
print("np.isnan  ->", np.isnan(a), "|", a[~np.isnan(a)])
print("\nnp.nan == np.nan ->", np.nan == np.nan, " NaN é diferente até de si mesmo")

Essa última linha é o motivo de existir `isna()` no pandas e `np.isnan()` aqui: você
**não pode** testar nulo com `== NaN`, porque nunca dá `True`.

**✏️ Exercícios rápidos**


1. Em `a = np.array([1.0, np.nan, 3.0, np.nan])`, quantos nulos há?
2. Calcule a média ignorando os nulos, de duas formas: com máscara e com `np.nanmean`.
3. Troque os nulos por zero sem usar `for`.

In [ ]:
a = np.array([1.0, np.nan, 3.0, np.nan])

# 1)


# 2)


# 3)

---
## 8. View × cópia

Fatiar um array **não copia**: devolve uma janela para a mesma memória. É rápido, e é uma
armadilha.

In [ ]:
a = np.arange(5)
fatia = a[1:4]
fatia[0] = 99
print(a, "<- a original mudou: fatia é VIEW")

b = np.arange(5)
copia = b[1:4].copy()
copia[0] = 99
print(b, "<- intacto")

print("\nindexação booleana já devolve cópia:", b[b > 2].base is None)

No pandas você não vê isso porque a partir da versão 3 ele sempre copia. Aqui a
escolha é sua, e é por isso que `.copy()` aparece tanto em código de NumPy.

---
## 9. Aleatório com semente

In [ ]:
rng = np.random.default_rng(42)        # gerador com semente: reprodutível
print(rng.random(3).round(3))
print(rng.integers(0, 10, 5))
print(rng.normal(100, 15, 4).round(1))
print(rng.choice(["loja", "site"], 5, p=[0.3, 0.7]))

print("\nmesma semente, mesmo resultado:", np.random.default_rng(42).random(3).round(3))

> Sempre com semente. Sem ela, o número muda a cada execução e você não consegue
> reproduzir o resultado nem depurar o que deu errado.

---
## 10. NumPy dentro do pandas

Não é analogia: é literalmente o que está lá dentro.

In [ ]:
s = pd.Series([10, 20, 30])
print(type(s.to_numpy()).__name__, s.to_numpy(), s.to_numpy().dtype)

df = pd.DataFrame({"a": [1, 2], "b": [3.0, 4.0]})
print("\n", df.to_numpy(), "<- vira um array só, com dtype comum a todas")
print("\nfunção do numpy direto na coluna:", np.sqrt(pd.Series([1.0, 4.0, 9.0])).tolist())
print("np.where numa coluna:", np.where(s > 15, "alto", "baixo"))

---
## Exercício em aula

Sem `for` e sem `if`. Tudo vetorizado.

1. Gere 1000 valores com `rng.normal(500, 200, 1000)` e semente 7. Quantos são negativos?
   Troque os negativos por zero **em uma linha**.
2. Quantos passam de 800? Qual o total desses?
3. Monte um array de comissões: 5% até 500, 8% acima. Use `np.where`. Some o total.
4. Monte uma matriz 12×5 (meses × vendedores) com `rng.integers(10_000, 40_000, (12, 5))`.
   Total por vendedor? Por mês? Qual vendedor teve o maior total, e em qual mês
   aconteceu a maior venda isolada?

In [ ]:
rng = np.random.default_rng(7)
vendas = rng.normal(500, 200, 1000)
matriz = np.random.default_rng(7).integers(10_000, 40_000, (12, 5))

print(vendas[:5].round(1), "|", matriz.shape)

# seu código aqui

## Para casa

1. Refaça o benchmark da seção 2 com `n = 100_000` e `n = 10_000_000`. A razão entre o
   `for` e o NumPy é constante, ou cresce com o tamanho?
2. Sem usar `np.median`: ordene com `np.sort` e calcule a mediana tratando os casos par e
   ímpar. Compare com `np.median`.
3. `np.unique(a, return_counts=True)` devolve os valores distintos e as contagens. Use
   para montar a contagem de frequências de um array de texto, e confira o resultado
   contra `pd.Series(a).value_counts()`.
4. Normalize a matriz 12×5 do exercício 4 para que cada **coluna** some 1. Cuidado com o
   `axis`, e repare em como o broadcasting resolve a divisão.

In [ ]:
# seu código aqui